In [1]:
import torch 
import numpy as np 
import h5py
import os
from pathlib import Path
import importlib
import IPython.display as ipd

import src.spatial_attn_lightning as binaural_lightning 
import yaml
from pytorch_lightning import Trainer, seed_everything

os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

# torch.set_float32_matmul_precision('medium')
# torch.backends.cuda.matmul.allow_tf32 = True
# torch.backends.cudnn.allow_tf32 = True

In [4]:
# import sys
# sys.path.append('/om2/user/annesyab/summer2022/CI_Model_HearingImpairment/cochlear_implant_param_infer/utils')
# import util_cimodel_soundcoding_master

    
# class AudioToCIRep(torch.nn.Module):
#     """
#     Converts audio to cochlear implant nervegram
#     """
#     def __init__(self, cgram_kwargs={}):
#         super(AudioToCIRep, self).__init__()
#         self.cgram_kwargs = cgram_kwargs

#         # Compression is applied as a separate transform to be consistent with Spectrograms
#         ci_simulator = util_cimodel_soundcoding_master.SequentialCIModel(self.cgram_kwargs)

#         self.CISimulator = ci_simulator

#     def forward(self, foreground_wav, background_wav):
#         """
#         Args:
#             foreground_wav (torch.Tensor): the waveform that will be used as
#                 the foreground audio sample (usually speech)
#             background_wav (torch.Tensor): the waveform that will be used as
#                 the background audio sample
#         """
#         del background_wav
#         if len(foreground_wav.shape)<2:
#             foreground_wav = torch.unsqueeze(foreground_wav,dim=0) #input has to be of shape batch X samples

#         foreground_ci_nervegram = self.CISimulator(foreground_wav)

#         return foreground_ci_nervegram, None

In [7]:
config_path = "config/ci_attention_models/ci_word_task_v09_4MGB.yaml"
config = yaml.safe_load(open(config_path, 'r'))

config['num_workers'] = 2
config['hparas']['batch_size'] = 32
# config['audio']['rep_kwargs']['rep_on_gpu'] = True
# print(f"Default lr is {config['hparas']['lr']}")




In [8]:
seed_everything(0)
importlib.reload(binaural_lightning)
module = binaural_lightning.BinauralAttentionModule(config)

[rank: 0] Seed set to 0


Using explicit dim specification for demeaning in audio transforms
Using BinauralAuditoryAttentionCNN
v08 True
num_classes={'num_words': 800}
Model performing word task
Conv block order: LN -> Conv -> ReLU
fc_attn: True
coch_affine: True


NotImplementedError: Audio Representation of type cochlear_implant is not implemented

In [4]:
# kaiming_init(module, mode='fan_in', init_type='uniform')

In [5]:
trainer = Trainer(
    precision="32",
    limit_val_batches=0.0,
    num_nodes=1,
    # benchmark=True,
    devices=1, # was gpus=1,
    # detect_anomaly=True,
    # strategy="ddp_notebook",
    accelerator="gpu",
)
trainer.fit(module)

/om2/user/imgriff/conda_envs/pytorch_2/lib/python3.11/site-packages/lightning_fabric/plugins/environments/slurm.py:191: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /om2/user/imgriff/conda_envs/pytorch_2/lib/python3.1 ...
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
/om2/user/imgriff/conda_envs/pytorch_2/lib/python3.11/site-packages/pytorch_lightning/loops/utilities.py:73: `max_epochs` was not set. Setting it to 1000 epochs. To train without an epoch limit, set `max_epochs=-1`.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name             | Type                         | Params
------------------------------------------------------------------
0 | audio_transforms | AudioCompose                 | 0     
1 | model            | BinauralAuditoryAtten

Using v06 dataset
/om/scratch/Sun/imgriff/datasets/spatial_audio_pipeline/assets/dataset_binaural_attn/v08
Using 0.1 cue free data
Using gender balanced training 4M set
cue type: mixed
mixture_percentages={'voice_only': 0.5, 'voice_and_location': 0.5}
913 files in train concat dataset
len training set = 114125
Epoch 0:   0%|          | 74/114125 [03:34<91:56:05,  0.34it/s, v_num=3.86e+7, train_loss=6.670, grad_norm=1.160]

/om2/user/imgriff/conda_envs/pytorch_2/lib/python3.11/site-packages/pytorch_lightning/trainer/call.py:54: Detected KeyboardInterrupt, attempting graceful shutdown...


In [ ]:
## Check dataset 

dataset = module.dataset(**config['corpus'], mode='train')

In [ ]:
cue, target, distractor, labels = dataset[0]

In [ ]:
aud_transforms = module.audio_transforms

In [ ]:
ix = 1080

cue, target, distractor, labels = dataset[ix]

scene, _ = aud_transforms(target, distractor)

print("cue")
ipd.display(ipd.Audio(cue[0], rate=44100, normalize=False))
print("target")
ipd.display(ipd.Audio(target[0], rate=44100, normalize=False))
print("distractor")
ipd.display(ipd.Audio(distractor[0], rate=44100, normalize=False))
print("scene")
ipd.display(ipd.Audio(scene[0], rate=44100, normalize=False))